# 04 - Seleccion de Variables: Correlacion y Mapas de Calor

---

## Contenido

1. Por que seleccionar variables
2. Correlacion: concepto y tipos
3. Matriz de correlacion
4. Mapas de calor (Heatmaps)
5. Seleccion de variables por umbral de correlacion
6. Interpretacion de coeficientes en RL
7. Ejemplo completo con el dataset Iris

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set()

## 1. Por que seleccionar variables

Como vimos en el notebook anterior, las funciones base le dan mayor flexibilidad a nuestra regresion lineal, pero se corre el riesgo de **sobreentrenar** el modelo.

La seleccion de variables busca responder la pregunta: **cuales de todas las variables disponibles realmente contribuyen a predecir la variable objetivo?**

Razones para seleccionar variables:

- **Reducir sobreajuste:** menos variables = menos ruido capturado
- **Mejorar interpretabilidad:** modelos mas simples son mas faciles de entender
- **Reducir tiempo de computo:** menos dimensiones = mas rapido
- **Evitar multicolinealidad:** variables altamente correlacionadas causan inestabilidad en los coeficientes

---

## 2. Correlacion: concepto y tipos

La **correlacion** es cualquier relacion estadistica, ya sea causal o no, entre dos o mas variables aleatorias. Mide como se relacionan unas con las otras.

Comunmente nos referimos al **grado en que un par de variables estan relacionadas linealmente**, medido por el coeficiente de correlacion $r$ (o $\rho$).

### Coeficiente de correlacion de Pearson

$$r_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

Propiedades:

| Valor de $r$ | Interpretacion |
|---|---|
| $r = 1$ | Correlacion positiva perfecta |
| $0.5 < r < 1$ | Correlacion positiva fuerte |
| $0 < r < 0.5$ | Correlacion positiva debil |
| $r = 0$ | Sin correlacion lineal |
| $-0.5 < r < 0$ | Correlacion negativa debil |
| $-1 < r < -0.5$ | Correlacion negativa fuerte |
| $r = -1$ | Correlacion negativa perfecta |

Las correlaciones son utiles porque pueden indicar una **relacion predictiva** entre variables que puede ser explotada en la practica.

### Ejemplo visual de correlaciones

In [ ]:
# Generar datos con diferentes niveles de correlacion
rng = np.random.RandomState(42)
n = 100

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

# r ~ 1
x1 = rng.randn(n)
y1 = x1 + 0.1 * rng.randn(n)
axes[0].scatter(x1, y1, alpha=0.6, color='steelblue')
axes[0].set_title(f'r = {np.corrcoef(x1, y1)[0,1]:.2f}\n(Positiva fuerte)', fontsize=11)

# r ~ 0.5
y2 = 0.5 * x1 + 0.8 * rng.randn(n)
axes[1].scatter(x1, y2, alpha=0.6, color='orange')
axes[1].set_title(f'r = {np.corrcoef(x1, y2)[0,1]:.2f}\n(Positiva debil)', fontsize=11)

# r ~ 0
y3 = rng.randn(n)
axes[2].scatter(x1, y3, alpha=0.6, color='green')
axes[2].set_title(f'r = {np.corrcoef(x1, y3)[0,1]:.2f}\n(Sin correlacion)', fontsize=11)

# r ~ -1
y4 = -x1 + 0.1 * rng.randn(n)
axes[3].scatter(x1, y4, alpha=0.6, color='red')
axes[3].set_title(f'r = {np.corrcoef(x1, y4)[0,1]:.2f}\n(Negativa fuerte)', fontsize=11)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Diferentes niveles de correlacion', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

---

## 3. Matriz de correlacion

La **matriz de correlacion** es una tabla simetrica que muestra el coeficiente de correlacion entre cada par de variables. Pandas la calcula facilmente con `DataFrame.corr()`.

### Ejemplo con un dataset real

Usaremos el dataset de generacion de energia de una planta de ciclo combinado (Combined Cycle Power Plant, CCPP) para ejemplificar:

In [ ]:
# Cargar el dataset CCPP
url = 'https://raw.githubusercontent.com/Alveuz/RandomDataSets/master/CombinedCyclePowerGen/Folds5x2_pp.csv'
ccpp_df = pd.read_csv(url)

print('Dataset de Generacion de Energia (CCPP)')
print(f'Dimensiones: {ccpp_df.shape}')
print()
print('Variables:')
print('  - AT: Temperatura ambiente (C)')
print('  - V:  Exhaust Vacuum (cm Hg)')
print('  - AP: Presion atmosferica (mbar)')
print('  - RH: Humedad relativa (%)')
print('  - PE: Energia generada (MW) [variable objetivo]')
print()
ccpp_df.head()

In [ ]:
# Calcular la matriz de correlacion
corr_matrix = ccpp_df.corr()
print('Matriz de correlacion:')
print(corr_matrix.round(3))

---

## 4. Mapas de Calor (Heatmaps)

Un **mapa de calor** es la representacion visual de la matriz de correlacion. Permite identificar rapidamente que variables estan fuertemente correlacionadas entre si y con la variable objetivo.

In [ ]:
# Mapa de calor del dataset CCPP
fig, ax = plt.subplots(figsize=(8, 6))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mascara triangular superior

sns.heatmap(corr_matrix,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            mask=mask,
            square=True,
            linewidths=1,
            ax=ax)

ax.set_title('Mapa de Calor: Correlaciones del dataset CCPP', fontsize=13)
plt.tight_layout()
plt.show()

---

## 5. Seleccion de variables por umbral de correlacion

Usando la matriz de correlacion, podemos determinar que variables usar para predecir la variable objetivo. Por ejemplo, podemos establecer un umbral de $|r| > 0.5$:

In [ ]:
# Correlacion de cada variable con PE (variable objetivo)
target_corr = corr_matrix['PE'].drop('PE')
print('Correlacion con PE (energia generada):')
print(target_corr.round(4))
print()

# Aplicar umbral
umbral = 0.5
selected = target_corr[target_corr.abs() > umbral]
print(f'Variables con |r| > {umbral}:')
print(selected.round(4))
print()
print(f'Variables seleccionadas: {list(selected.index)}')

---

## 6. Interpretacion de coeficientes en RL

Los valores de los coeficientes de un modelo de RL pueden ser utilizados para indicar la **naturaleza entre las variables independientes y la variable dependiente**.

Respecto a los coeficientes, el **signo** y la **magnitud** indican diferentes cosas:

- **Signo:** el sentido de la correlacion entre cada variable independiente y la dependiente.
  - Positivo: al aumentar la variable independiente, la dependiente tiende a aumentar.
  - Negativo: al aumentar la variable independiente, la dependiente tiende a disminuir.

- **Magnitud:** la fuerza de la relacion.
  - Para que la magnitud sea comparable entre variables, estas deben estar **estandarizadas**.

Veamos un ejemplo con el dataset CCPP:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

# Preparar datos
X_ccpp = ccpp_df.iloc[:, 0:4]
y_ccpp = ccpp_df['PE']

X_train, X_test, y_train, y_test = train_test_split(
    X_ccpp, y_ccpp, test_size=0.33, random_state=100
)

# Estandarizar
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

# Ajustar modelo con statsmodels para obtener p-valores
X_train_sm = sm.add_constant(X_train_scl)
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

---

## 7. Ejemplo completo con el dataset Iris

Apliquemos todo lo aprendido al dataset Iris. El objetivo sera predecir `petal_width` seleccionando las variables mas relevantes mediante correlacion.

In [ ]:
# Cargar datos
iris = sns.load_dataset('iris')

# Solo columnas numericas
iris_num = iris.select_dtypes(include=[np.number])

# Matriz de correlacion
corr_iris = iris_num.corr()

# Mapa de calor
fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_iris, dtype=bool))
sns.heatmap(corr_iris,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            mask=mask,
            square=True,
            linewidths=1,
            ax=ax)
ax.set_title('Mapa de Calor: Correlaciones del dataset Iris', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlacion con la variable objetivo: petal_width
target = 'petal_width'
target_corr_iris = corr_iris[target].drop(target)

print(f'Correlacion con {target}:')
print(target_corr_iris.round(4))
print()

# Seleccionar variables con |r| > 0.5
umbral = 0.5
selected_vars = target_corr_iris[target_corr_iris.abs() > umbral]
print(f'Variables seleccionadas (|r| > {umbral}):')
print(selected_vars.round(4))

In [ ]:
# Construir modelo solo con variables seleccionadas
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error

selected_features = list(selected_vars.index)
print(f'Usando variables: {selected_features}')

X_sel = iris[selected_features].values
y_sel = iris[target].values

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y_sel, test_size=0.30, random_state=42
)

# Modelo con estandarizacion + polinomio grado 2 + RL
model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(2),
    LinearRegression()
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Metricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'\nResultados del modelo:')
print(f'  R^2:  {r2:.4f}')
print(f'  RMSE: {rmse:.4f}')

In [ ]:
# Grafica de predicciones vs reales
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, color='steelblue', alpha=0.7, edgecolors='black', linewidth=0.5)
ax.plot([0, 2.5], [0, 2.5], 'r--', linewidth=1.5, label='Linea ideal')
ax.set_xlabel('Valor Real (petal_width)', fontsize=12)
ax.set_ylabel('Valor Predicho (petal_width)', fontsize=12)
ax.set_title(f'Predicciones del modelo (R2={r2:.3f})', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---

## Resumen

En este notebook aprendimos:

1. La **seleccion de variables** reduce sobreajuste, mejora la interpretabilidad y evita multicolinealidad.
2. La **correlacion de Pearson** mide la relacion lineal entre dos variables ($r \in [-1, 1]$).
3. La **matriz de correlacion** y los **mapas de calor** permiten visualizar todas las correlaciones simultaneamente.
4. Podemos usar un **umbral de correlacion** (e.g., $|r| > 0.5$) para seleccionar variables relevantes.
5. Los **coeficientes** del modelo RL (signo y magnitud) indican la naturaleza de la relacion entre variables.
6. Aplicamos estas tecnicas al dataset Iris para seleccionar variables y construir un modelo predictivo.

En el siguiente notebook veremos las **Metricas de Evaluacion: R^2 y MSE**.